# Inspect PartEdit-Bench For Part-Level TDM Localization

This notebook checks whether PartEdit-Bench can support Harry Yang's requested pilot study: 10-15 cases balanced by target part size, with ground-truth masks for evaluating Follow-Your-Shape TDM localization.

## Goal

Before renting a GPU or running Follow-Your-Shape, confirm the dataset fields, image/mask formats, prompt structure, and mask-area distribution. The notebook should produce a small candidate table, not download or commit the full dataset into the repository.

## Setup

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from datasets import get_dataset_config_names, load_dataset
from PIL import Image


def find_repo_root(start):
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "core").exists():
            return candidate
    raise FileNotFoundError(f"Could not find repository root from {start}")


REPO_ROOT = find_repo_root(Path.cwd())
DATASET_ID = "Aleksandar/PartEdit-Bench"
OUTPUT_DIR = REPO_ROOT / "core" / "data" / "partedit_subset"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PREFERRED_SPLIT = "real"
IMAGE_FIELD = "original_image"
EDITED_REFERENCE_FIELD = "partedit"
MASK_FIELD = "gt_mask"
SOURCE_PROMPT_FIELD = "prompt_original"
TARGET_PROMPT_FIELD = "p2p_prompt"
REFERENCE_TARGET_PROMPT_FIELD = "prompt_changed"
PART_FIELD = "part"

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 160)

print(f"Repository root: {REPO_ROOT}")
print(f"Subset output directory: {OUTPUT_DIR}")

## 1. Inspect Dataset Configs And Splits

In [ ]:
configs = get_dataset_config_names(DATASET_ID)
configs

In [ ]:
dataset_kwargs = {}
if configs:
    dataset_kwargs["name"] = configs[0]

dataset = load_dataset(DATASET_ID, **dataset_kwargs)
dataset

## 2. Inspect Actual Schema And Example Content

In [ ]:
split_name = PREFERRED_SPLIT if PREFERRED_SPLIT in dataset else list(dataset.keys())[0]
split = dataset[split_name]

print("split:", split_name)
print("num rows:", len(split))
split.features

In [ ]:
sample = split[0]

field_rows = []
for field_name, value in sample.items():
    if isinstance(value, Image.Image):
        preview = f"PIL.Image size={value.size} mode={value.mode}"
    else:
        preview = value
    field_rows.append({
        "field": field_name,
        "python_type": type(value).__name__,
        "preview": preview,
    })

pd.DataFrame(field_rows)

In [ ]:
task_fields = [
    "id",
    "class_name",
    "subject",
    "part",
    "edit",
    "seed",
    SOURCE_PROMPT_FIELD,
    TARGET_PROMPT_FIELD,
    REFERENCE_TARGET_PROMPT_FIELD,
    "p2p_template",
    "instructp2p_edit1",
    "instructp2p_edit2",
    "instructp2p_edit3",
]

pd.DataFrame([
    {"field": field, "value": sample[field]}
    for field in task_fields
    if field in sample
])

## 3. Set Follow-Your-Shape Field Mapping

PartEdit-Bench already provides the fields needed for this diagnostic. `prompt_original` and `prompt_changed` map directly to Follow-Your-Shape's source and target prompts, while `gt_mask` is used only for evaluation.

In [ ]:
field_mapping = pd.DataFrame([
    {"experiment_role": "source image", "dataset_field": IMAGE_FIELD, "used_for": "Follow-Your-Shape input"},
    {"experiment_role": "source prompt", "dataset_field": SOURCE_PROMPT_FIELD, "used_for": "Follow-Your-Shape input"},
    {"experiment_role": "target prompt", "dataset_field": TARGET_PROMPT_FIELD, "used_for": "Follow-Your-Shape input; p2p_prompt is preferred because it usually states the local part edit explicitly"},
    {"experiment_role": "original target prompt", "dataset_field": REFERENCE_TARGET_PROMPT_FIELD, "used_for": "reference only; often describes the full target image rather than the local edit"},
    {"experiment_role": "ground-truth part mask", "dataset_field": MASK_FIELD, "used_for": "localization evaluation only"},
    {"experiment_role": "target part label", "dataset_field": PART_FIELD, "used_for": "case description and grouping"},
    {"experiment_role": "PartEdit reference image", "dataset_field": EDITED_REFERENCE_FIELD, "used_for": "optional qualitative reference"},
])

field_mapping


In [ ]:
required_fields = [
    IMAGE_FIELD,
    MASK_FIELD,
    SOURCE_PROMPT_FIELD,
    TARGET_PROMPT_FIELD,
    PART_FIELD,
]
missing_fields = [field for field in required_fields if field not in sample]

if missing_fields:
    raise KeyError(f"Missing required fields: {missing_fields}. Available fields: {list(sample.keys())}")

print("Required PartEdit-Bench fields are available.")

## 4. Visual Check One Example

In [ ]:
def as_mask_array(value):
    if isinstance(value, Image.Image):
        arr = np.asarray(value.convert("L"))
    else:
        arr = np.asarray(value)
    if arr.ndim == 3:
        arr = arr[..., 0]
    return arr > 0


def mask_area_ratio(mask_value):
    mask = as_mask_array(mask_value)
    return float(mask.mean())


image = sample[IMAGE_FIELD]
mask = as_mask_array(sample[MASK_FIELD])

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(image)
axes[0].set_title("original_image")
axes[1].imshow(mask, cmap="gray")
axes[1].set_title(f"gt_mask ratio={mask.mean():.3f}")
axes[2].imshow(image)
axes[2].imshow(mask, alpha=0.35, cmap="Reds")
axes[2].set_title("mask overlay")
for ax in axes:
    ax.axis("off")
plt.tight_layout()

## 5. Count Parts Before Any Selection

The pilot should be driven by valid part labels, not by a large edit-based candidate pool. The first task is to inspect how many cases exist per part and which parts are rich enough for manual prompt-level review.


In [ ]:
def build_part_table(dataset_dict):
    rows = []
    for split_name in dataset_dict.keys():
        split = dataset_dict[split_name]
        for idx in range(len(split)):
            row = split[idx]
            rows.append({
                "case_uid": f"{split_name}_{idx:04d}",
                "dataset_split": split_name,
                "dataset_index": idx,
                "id": row["id"],
                "class_name": row.get("class_name", ""),
                "subject": row.get("subject", ""),
                "part": row.get(PART_FIELD, ""),
                "edit": row.get("edit", ""),
                "source_prompt": row.get(SOURCE_PROMPT_FIELD, ""),
                "target_prompt": row.get(TARGET_PROMPT_FIELD, ""),
                "reference_target_prompt": row.get(REFERENCE_TARGET_PROMPT_FIELD, ""),
                "mask_area_ratio": mask_area_ratio(row[MASK_FIELD]),
            })
    return pd.DataFrame(rows).sort_values("mask_area_ratio").reset_index(drop=True)


part_table = build_part_table(dataset)
part_table.head()


In [ ]:
part_counts = (
    part_table.groupby("part", as_index=False)
    .size()
    .rename(columns={"size": "count"})
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)
part_counts


In [ ]:
part_counts_by_split = (
    part_table.groupby(["dataset_split", "part"], as_index=False)
    .size()
    .rename(columns={"size": "count"})
    .sort_values(["dataset_split", "count"], ascending=[True, False])
    .reset_index(drop=True)
)
part_counts_by_split


## 6. Build A Hard-Filtered Prompt Review Pool

This section does not decide whether a case is scientifically valid. It only removes cases that are mechanically unusable, then prepares a table for manual prompt-mask review. The key manual question is: does the difference between `prompt_original` and `prompt_changed` explicitly point to the same local part as `gt_mask`?

In [ ]:
def nonempty_text(value):
    return isinstance(value, str) and bool(value.strip())


hard_filter_table = part_table.copy()
hard_filter_table["has_nonempty_mask"] = hard_filter_table["mask_area_ratio"] > 0
hard_filter_table["has_source_prompt"] = hard_filter_table["source_prompt"].map(nonempty_text)
hard_filter_table["has_target_prompt"] = hard_filter_table["target_prompt"].map(nonempty_text)
hard_filter_table["has_changed_prompt"] = (
    hard_filter_table["source_prompt"].str.strip()
    != hard_filter_table["target_prompt"].str.strip()
)
hard_filter_table["has_part_label"] = hard_filter_table["part"].map(nonempty_text)

hard_filter_columns = [
    "has_nonempty_mask",
    "has_source_prompt",
    "has_target_prompt",
    "has_changed_prompt",
    "has_part_label",
]
hard_filter_table["passes_hard_filter"] = hard_filter_table[hard_filter_columns].all(axis=1)

hard_filter_summary = (
    hard_filter_table[hard_filter_columns + ["passes_hard_filter"]]
    .sum()
    .rename("num_passed")
    .reset_index()
    .rename(columns={"index": "check"})
)

hard_filter_summary


In [ ]:
hard_filtered_table = hard_filter_table[hard_filter_table["passes_hard_filter"]].copy()

hard_filtered_table["part_size"] = pd.qcut(
    hard_filtered_table["mask_area_ratio"].rank(method="first"),
    q=3,
    labels=["small", "medium", "large"],
)

hard_filtered_table[[
    "case_uid",
    "dataset_split",
    "dataset_index",
    "part_size",
    "mask_area_ratio",
    "class_name",
    "subject",
    "part",
    "edit",
    "source_prompt",
    "target_prompt",
    "reference_target_prompt",
]].head(20)


In [ ]:
hard_filtered_table.groupby(["dataset_split", "part_size"], observed=True).size().rename("num_cases").reset_index()

In [ ]:
PILOT_CASE_COUNT = 20
PART_SIZE_QUOTAS = {"small": 7, "medium": 7, "large": 6}

full_review_pool = (
    hard_filtered_table.assign(
        split_priority=hard_filtered_table["dataset_split"].map({"real": 0, "synth": 1}).fillna(2)
    )
    .sort_values(["part_size", "split_priority", "mask_area_ratio", "dataset_index"])
    .drop(columns=["split_priority"])
    .reset_index(drop=True)
)

pilot_parts = []
for part_size, quota in PART_SIZE_QUOTAS.items():
    pilot_parts.append(
        full_review_pool[full_review_pool["part_size"] == part_size].head(quota)
    )

review_pool = (
    pd.concat(pilot_parts, ignore_index=True)
    .sort_values(["part_size", "dataset_split", "mask_area_ratio", "dataset_index"])
    .reset_index(drop=True)
)
review_pool.insert(0, "review_index", range(len(review_pool)))

assert len(review_pool) == PILOT_CASE_COUNT, f"Expected {PILOT_CASE_COUNT} pilot cases, got {len(review_pool)}"

review_pool[[
    "review_index",
    "case_uid",
    "dataset_split",
    "dataset_index",
    "part_size",
    "mask_area_ratio",
    "part",
    "edit",
    "source_prompt",
    "target_prompt",
    "reference_target_prompt",
]]


### 6.1 Visual Prompt-Mask Review

Use this helper to inspect each candidate. Reject cases where the prompt implies a whole-object change while `gt_mask` marks only a part, such as `cat running -> dog running` with a head-only mask.

In [ ]:
def mask_array_and_display_mask(mask_value, image):
    raw_mask = as_mask_array(mask_value)
    image_width, image_height = image.size
    expected_shape = (image_height, image_width)
    resized_for_display = raw_mask.shape != expected_shape

    if resized_for_display:
        mask_image = Image.fromarray((raw_mask.astype(np.uint8) * 255), mode="L")
        mask_image = mask_image.resize(image.size, resample=Image.Resampling.NEAREST)
        display_mask = np.asarray(mask_image) > 0
    else:
        display_mask = raw_mask

    return raw_mask, display_mask, resized_for_display


def get_dataset_row(case_uid):
    case_row = review_pool.loc[review_pool["case_uid"] == case_uid]
    if case_row.empty:
        raise KeyError(f"Unknown case_uid: {case_uid}")
    case_row = case_row.iloc[0]
    return dataset[case_row["dataset_split"]][int(case_row["dataset_index"])], case_row


def get_case_uid_by_review_index(review_index):
    matches = review_pool.loc[review_pool["review_index"] == review_index, "case_uid"]
    if matches.empty:
        raise IndexError(f"review_index {review_index} is out of range 0-{len(review_pool) - 1}")
    return matches.iloc[0]


def show_review_case(case_uid):
    dataset_row, case_row = get_dataset_row(case_uid)
    image = dataset_row[IMAGE_FIELD]
    raw_mask, mask, resized_for_display = mask_array_and_display_mask(dataset_row[MASK_FIELD], image)
    reference = dataset_row[EDITED_REFERENCE_FIELD]

    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    resize_note = " | mask resized for display" if resized_for_display else ""
    fig.suptitle(
        f"index={case_row['review_index']} | {case_uid} | {case_row['dataset_split']} | "
        f"{case_row['part_size']} | ratio={case_row['mask_area_ratio']:.3f} | "
        f"part={case_row['part']}{resize_note}",
        fontsize=11,
    )
    axes[0].imshow(image)
    axes[0].set_title("original_image")
    axes[1].imshow(raw_mask, cmap="gray")
    axes[1].set_title(f"gt_mask raw {raw_mask.shape}")
    axes[2].imshow(image)
    axes[2].imshow(mask, alpha=0.35, cmap="Reds")
    axes[2].set_title(f"overlay on image {image.size}")
    axes[3].imshow(reference)
    axes[3].set_title("PartEdit reference")
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

    print("review_index:", int(case_row["review_index"]))
    print("case_uid:", case_uid)
    print("part:", dataset_row[PART_FIELD])
    print("edit:", dataset_row["edit"])
    print("source prompt:", dataset_row[SOURCE_PROMPT_FIELD])
    print("target prompt (p2p_prompt):", dataset_row[TARGET_PROMPT_FIELD])
    print("reference target prompt (prompt_changed):", dataset_row.get(REFERENCE_TARGET_PROMPT_FIELD, ""))
    print("p2p_template:", dataset_row.get("p2p_template", ""))
    print("instructp2p_edit1:", dataset_row.get("instructp2p_edit1", ""))
    print("instructp2p_edit2:", dataset_row.get("instructp2p_edit2", ""))
    print("instructp2p_edit3:", dataset_row.get("instructp2p_edit3", ""))
    print("raw mask shape:", raw_mask.shape, "image size:", image.size, "resized_for_display:", resized_for_display)


def show_review_index(review_index):
    case_uid = get_case_uid_by_review_index(review_index)
    show_review_case(case_uid)

In [ ]:
show_review_index(2)

### 6.2 Review Only The 20 Pilot Candidates

The pilot set above already uses `p2p_prompt` as the target prompt and is balanced across small, medium, and large part masks. Inspect only these 20 cases with `show_review_index(i)`. Add a case to `rejected_case_uids` only if the `p2p_prompt` is still unclear, mismatched with the mask, or not a local part edit.


In [ ]:
rejected_case_uids = [
    # Example:
    # "synth_0026",
]

reject_reasons = {
    # Example:
    # "synth_0026": "p2p_prompt is grammatically unclear and does not explicitly preserve the subject",
}

reviewed_table = review_pool.copy()
reviewed_table["prompt_valid"] = ~reviewed_table["case_uid"].isin(rejected_case_uids)
reviewed_table["review_reason"] = reviewed_table["case_uid"].map(reject_reasons).fillna("")

reviewed_table[[
    "review_index",
    "case_uid",
    "dataset_split",
    "part_size",
    "mask_area_ratio",
    "part",
    "edit",
    "prompt_valid",
    "review_reason",
    "source_prompt",
    "target_prompt",
    "reference_target_prompt",
]]


In [ ]:
accepted_prompt_cases = reviewed_table[reviewed_table["prompt_valid"]].copy()

accepted_prompt_cases.groupby(["part_size", "dataset_split"], observed=True).size().rename("num_cases").reset_index()


## 7. Export Follow-Your-Shape-Ready Pilot Data

This section freezes the reviewed pilot cases into a local file structure that can be consumed by `core/third_party/FollowYourShape/src/edit.py`. The exported images and masks are local artifacts and should not be committed; the manifest records the exact case IDs, prompts, masks, and suggested command arguments.


In [ ]:
import os
import shlex

PILOT_CASES_DIR = OUTPUT_DIR / "cases"
PILOT_MANIFEST_CSV = OUTPUT_DIR / "pilot_manifest.csv"
PILOT_MANIFEST_JSON = OUTPUT_DIR / "pilot_manifest.json"
FYS_RESULTS_DIR = REPO_ROOT / "core" / "results" / "follow_your_shape"


def normalize_part_label(part):
    normalized = str(part).strip().lower().replace("_", " ")
    aliases = {
        "carhood": "car hood",
        "car hood": "car hood",
        "carbody": "car body",
        "car body": "car body",
    }
    return aliases.get(normalized, normalized)


def repo_relative(path):
    return str(Path(path).resolve().relative_to(REPO_ROOT))


def path_from_fys_src(path):
    fys_src_dir = REPO_ROOT / "core" / "third_party" / "FollowYourShape" / "src"
    return os.path.relpath(Path(path).resolve(), start=fys_src_dir)


def save_binary_mask(mask_value, source_image, output_path):
    mask = as_mask_array(mask_value)
    mask_image = Image.fromarray((mask.astype(np.uint8) * 255), mode="L")
    if mask_image.size != source_image.size:
        mask_image = mask_image.resize(source_image.size, resample=Image.Resampling.NEAREST)
    mask_image.save(output_path)
    return mask_image


PILOT_CASES_DIR.mkdir(parents=True, exist_ok=True)
FYS_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

export_records = []
for _, case in accepted_prompt_cases.reset_index(drop=True).iterrows():
    case_uid = case["case_uid"]
    dataset_row = dataset[case["dataset_split"]][int(case["dataset_index"])]
    case_dir = PILOT_CASES_DIR / case_uid
    case_dir.mkdir(parents=True, exist_ok=True)

    source_path = case_dir / "source.png"
    mask_path = case_dir / "gt_mask.png"
    reference_path = case_dir / "partedit_reference.png"
    metadata_path = case_dir / "metadata.json"
    output_dir = FYS_RESULTS_DIR / case_uid
    vis_path = output_dir / "tdm"

    source_image = dataset_row[IMAGE_FIELD].convert("RGB")
    reference_image = dataset_row[EDITED_REFERENCE_FIELD].convert("RGB")
    source_image.save(source_path)
    save_binary_mask(dataset_row[MASK_FIELD], source_image, mask_path)
    reference_image.save(reference_path)

    source_prompt = str(case["source_prompt"])
    target_prompt = str(case["target_prompt"])
    normalized_part = normalize_part_label(case["part"])

    edit_args = [
        "--source_img_dir", path_from_fys_src(source_path),
        "--source_prompt", source_prompt,
        "--target_prompt", target_prompt,
        "--guidance", "2",
        "--num_steps", "15",
        "--front", "2",
        "--inject", "4",
        "--name", "flux-dev",
        "--offload",
        "--controlnet_type", "none",
        "--output_dir", path_from_fys_src(output_dir),
        "--vis_path", path_from_fys_src(vis_path),
    ]
    fys_command = (
        "cd "
        + shlex.quote("core/third_party/FollowYourShape/src")
        + " && python edit.py "
        + " ".join(shlex.quote(str(part)) for part in edit_args)
    )

    metadata = {
        "case_uid": case_uid,
        "dataset_id": DATASET_ID,
        "dataset_split": case["dataset_split"],
        "dataset_index": int(case["dataset_index"]),
        "id": int(case["id"]),
        "class_name": case.get("class_name", ""),
        "subject": case.get("subject", ""),
        "part": case["part"],
        "normalized_part": normalized_part,
        "part_size": str(case["part_size"]),
        "mask_area_ratio": float(case["mask_area_ratio"]),
        "edit": case["edit"],
        "source_prompt": source_prompt,
        "target_prompt": target_prompt,
        "reference_target_prompt": case["reference_target_prompt"],
        "source_image": repo_relative(source_path),
        "gt_mask": repo_relative(mask_path),
        "partedit_reference": repo_relative(reference_path),
        "follow_your_shape_output_dir": repo_relative(output_dir),
        "follow_your_shape_vis_path": repo_relative(vis_path),
        "follow_your_shape_command": fys_command,
        "oracle_mask_arg": f"--mask_path {shlex.quote(path_from_fys_src(mask_path))}",
    }
    metadata_path.write_text(json.dumps(metadata, indent=2, ensure_ascii=False) + "\n")
    export_records.append(metadata)

pilot_manifest = pd.DataFrame(export_records)
pilot_manifest.to_csv(PILOT_MANIFEST_CSV, index=False)
PILOT_MANIFEST_JSON.write_text(json.dumps(export_records, indent=2, ensure_ascii=False) + "\n")

print(f"Exported {len(pilot_manifest)} cases to {PILOT_CASES_DIR}")
print(f"Wrote {repo_relative(PILOT_MANIFEST_CSV)}")
print(f"Wrote {repo_relative(PILOT_MANIFEST_JSON)}")

pilot_manifest[[
    "case_uid",
    "dataset_split",
    "part_size",
    "normalized_part",
    "mask_area_ratio",
    "source_image",
    "gt_mask",
    "source_prompt",
    "target_prompt",
]]
